In [9]:
import dolfinx
import torch
import torch_geometric as tg
from Training_utils import train_set as dataset
from FEniCSx_PyTorch_interface import Data_to_solver, fem_solver
num = 8

fs, G = Data_to_solver(num, train=False)
x = torch.tensor(fs.yh.x.array.reshape(-1,1), dtype=torch.float64, requires_grad=True)

loss_fn2 = fem_solver(fs)
loss_fn3 = torch.nn.functional.mse_loss

width = 4
depth = 128
model = tg.nn.models.GAT(in_channels=10, hidden_channels=width, out_channels=1, num_layers=depth, v2=True)
#model = tg.nn.models.MLP(in_channels=10, hidden_channels=width, out_channels=1, num_layers=depth)
optimizer = torch.optim.Adam([x], lr=0.001)

loss_fn2(x)

tensor(0.7742, dtype=torch.float64,
       grad_fn=<FEniCSx_PyTorch_interfaceBackward>)

In [17]:

#optimizer = torch.optim.LBFGS(params=[x], lr=1, max_iter=20, history_size=20, tolerance_grad=1e-16, tolerance_change=1e-16, line_search_fn="strong_wolfe")
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer)
optimizer = torch.optim.Adam(params=[x], lr = 0.001)


0.022249663120206375
0.14338750116639137
0.09160499620380654
0.09240413463842909
0.09983937156392807
0.08897206709354405
0.07211736233665661
0.0570942261507534
0.053694857617787306
0.050805636270568284
0.05508123277842964
0.05433934578988719
0.045689044561418306
0.036812737124058796
0.035668038377608936
0.04103371315520116
0.03628897385287339
0.04024074553264345
0.03444849145190484
0.03451742663491751
0.033120576114411396
0.030828177074600324
0.031107785786618165
0.030514989703278077
0.029203031889980822
0.02936015165245605
0.029310466772370564
0.029968261669652767
0.02830519678058431
0.026933942784720756
0.026510075194043574
0.026985174486771166
0.027056704114052622
0.02565582930190596
0.025443227482363148
0.025434113982231204
0.02473092189741037
0.02428711255867666
0.023903904903416047
0.023448907419754774
0.023251812546694978
0.02339672393233704
0.02317955462867257
0.022802354379277075
0.02273112966856997
0.022696938301826508
0.02263313511830557
0.02266279069189868
0.022731130009675

In [19]:

def closure():
    optimizer.zero_grad()
    loss = loss_fn2(x)
    loss.backward()
    return loss
for epoch in range(5000):
    loss = optimizer.step(closure)
    scheduler.step(loss)
    print(loss.item())

0.01536130602561927
0.014868882475763467
0.015452737066044024
0.014572479140592183
0.015114523298538306
0.016211207108523026
0.015989747982779065
0.017971603196626204
0.01598325252715989
0.016305313678197673
0.015655031997634725
0.01593782152688003
0.015854678657825166
0.01546800205991811
0.015656891169323103
0.015262812553173083
0.01525737220690496
0.014922854205780012
0.01497333221807008
0.014796623083315561
0.01480539502672287
0.014806858215423336
0.014741107459554825
0.014751491712097043
0.014689469280161621
0.014613873567414018
0.014488310484499844
0.014522636445407466
0.014513415350651064
0.014607490912313502
0.014972039167214332
0.015654157463447914
0.01629929358549793
0.016171236008166206
0.017088445116040258
0.015486357050775222
0.015033654475177216
0.014778066069223533
0.014837667595617124
0.014912163660674202
0.01570085829192647
0.017556151225401277
0.01770168810713937
0.014973048816977128
0.014559065756962926
0.016182593867030443
0.015790201203038656
0.014467104320704086
0.

In [18]:
from FEniCSx_solver import fem_plotter_grid

grid = fem_plotter_grid(fs.Wh)
grid.add_data(fs.uh)
import pyvista as pv

p = pv.Plotter()

p.add_mesh(grid.grid.warp_by_scalar(), show_edges=True)
p.show()

Widget(value='<iframe src="http://localhost:49887/index.html?ui=P_0x3508b51d0_4&reconnect=auto" class="pyvista…

In [ ]:
def unpack_params(model, x):
    pointer = 0
    for p in model.parameters():
        num = p.numel()
        new_val = x[pointer:pointer+num].reshape(p.shape)
        p.data = torch.tensor(new_val)
        pointer += num




In [ ]:
torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/MIX_self_supervised.pth")